## Load the dataset 

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv("C:/Projects/03_Advanced_house_price _predictor/data/raw/kc_house_data.csv")

## Dataset Overview

In [4]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [5]:
df.shape

(21613, 21)

In [6]:
df.columns

Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long  

### Initial Observation
- This data set has **21613 rows** and **21 columns**.
- There are **No Missing values**.
- This dataset has a mix of **categorical**, **numerical** and **date-related feature**.


## Data Cleaning and Preprocessing

In [8]:
# Creating the working copy
df_clean = df.copy()

In [9]:
# Checking missing values
missing = df_clean.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing

Series([], dtype: int64)

In [11]:
# Standardize the column names 
df_clean.columns = (
    df_clean.columns.str.strip()
             .str.lower()
             .str.replace(" ", "_")
)
df_clean.columns


Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='object')

In [ ]:
# Check Duplicates
df_clean.duplicated().sum()
# No duplicate are found

np.int64(0)

In [13]:
# Convert date column to date time
df_clean["date"] = pd.to_datetime(
    df_clean["date"].astype(str).str[:8],
    format="%Y%m%d",
    errors="coerce"
)

df_clean[["date"]].head()

,date
0,2014-10-13
1,2014-12-09
2,2015-02-25
3,2014-12-09
4,2015-02-18


In [ ]:
# Fix the zipcode datatype and check it again
df_clean["zipcode"] = df_clean["zipcode"].astype(str)
df_clean.dtypes
#zipcode should be categorical(not numerical) because there is no meaningful math relationship between zip codes.

id                        int64
date             datetime64[ns]
price                   float64
bedrooms                  int64
bathrooms               float64
sqft_living               int64
sqft_lot                  int64
floors                  float64
waterfront                int64
view                      int64
condition                 int64
grade                     int64
sqft_above                int64
sqft_basement             int64
yr_built                  int64
yr_renovated              int64
zipcode                  object
lat                     float64
long                    float64
sqft_living15             int64
sqft_lot15                int64
dtype: object

In [18]:
# Remove impossible values
# 1. Price must be positive
df_clean = df_clean[df_clean["price"] > 0].copy()


In [19]:
# 2. Living area must be positive
df_clean = df_clean[df_clean["sqft_living"] > 0].copy()


In [20]:
# bedrooms and bathrooms must be positive
df_clean = df_clean[(df_clean["bedrooms"] >= 0) & (df_clean["bathrooms"] >= 0)].copy()


In [21]:
# Remove the outliers

def clip_outliers_iqr(series, low_q=0.01, high_q=0.99):
    """Clip extreme values using quantiles (winsorization)."""
    lo = series.quantile(low_q)
    hi = series.quantile(high_q)
    return series.clip(lower=lo, upper=hi)

cols_to_clip = ["price", "sqft_living", "sqft_lot", "sqft_living15", "sqft_lot15"]

for col in cols_to_clip:
    if col in df_clean.columns:
        df_clean[col] = clip_outliers_iqr(df_clean[col], 0.01, 0.99)

df_clean[cols_to_clip].describe().T

,count,mean,std,min,25%,50%,75%,max
price,21613.0,532888.126735,317438.089268,153500.36,321950.0,450000.0,645000.0,1964400.0
sqft_living,21613.0,2070.894721,872.755768,720.00,1427.0,1910.0,2550.0,4978.8
sqft_lot,21613.0,13985.009996,27703.147345,1013.12,5040.0,7618.0,10688.0,213008.0
sqft_living15,21613.0,1982.897238,666.839300,950.00,1490.0,1840.0,2360.0,4080.0
sqft_lot15,21613.0,12004.317822,19872.532922,1191.24,5100.0,7620.0,10083.0,157687.0


In [23]:
# Save the cleaned data
df_clean.to_csv("C:/Projects/03_Advanced_house_price _predictor/data/cleaned/house_price_cleaned_data.csv", index=False)
print("Cleaned dataset saved")

Cleaned dataset saved
